# 03. Creating the CloudXeus Invoice Intelligence Agent (with an MCP Knowledge-Base Tool)

This notebook creates (or, on re-run, *versions*) a persistent **Prompt Agent** in the Azure AI Foundry Agent Service, named `CloudXeus-Invoice-Intelligence-Agent`. The agent is wired to the MCP knowledge-base tool whose *connection* we registered in `02_project_mcp_conn.ipynb`, and is given detailed system instructions describing how to reason over extracted invoice data and cross-reference it against CloudXeus's product knowledge base. **This notebook only defines and registers the agent — it does not run it.** The next notebook, `04_invoke_agent.ipynb`, invokes this exact agent by name (`AGENT_NAME = "CloudXeus-Invoice-Intelligence-Agent"`) via the `agent_reference` mechanism, so this notebook must be run at least once, successfully, before `04_invoke_agent.ipynb` will work — and `02_project_mcp_conn.ipynb` must have already registered the `cloudxeus-product-kb-mcp-connection` connection this agent's tool depends on.

**Difficulty: Advanced** (chains a Foundry project connection, an MCP tool definition, and a versioned persistent-agent registration together).

## Prerequisites

**Python packages**
```bash
pip install azure-ai-projects azure-identity python-dotenv
```
The repo root `requirements.txt` already pins `azure-ai-projects==2.2.0` and `azure-identity==1.25.3` for `11_azure_ai_foundry/`. `PromptAgentDefinition` and `MCPTool` are part of the newer Foundry **Agent Service** / MCP-tool preview surface — if you get an `ImportError`, upgrade first:
```bash
pip install --upgrade azure-ai-projects azure-ai-agents
```

**Azure resources**
- The same **Azure AI Foundry project** used in `11_azure_ai_foundry/` and in `02_project_mcp_conn.ipynb`.
- A **model deployment** in that project (e.g. a GPT-family chat model) to back the agent.
- The MCP connection created by `02_project_mcp_conn.ipynb` (`cloudxeus-product-kb-mcp-connection`, pointing at the `kb-cloudxeus-course-products` knowledge base) must already exist.

**Environment variables** (add to the root `.env` — reusing the existing `AZURE_AI_PROJECT_ENDPOINT` / `AZURE_AI_MODEL_DEPLOYMENT` pattern from `11_azure_ai_foundry/`):

```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-chat-model-deployment-name>
AZURE_AI_AGENT_NAME=CloudXeus-Invoice-Intelligence-Agent
AZURE_SEARCH_SERVICE_ENDPOINT=https://<your-search-service>.search.windows.net
AZURE_SEARCH_KNOWLEDGE_BASE_NAME=kb-cloudxeus-course-products
AZURE_AI_PROJECT_MCP_CONNECTION_NAME=cloudxeus-product-kb-mcp-connection
```

## What You'll Learn

- `PromptAgentDefinition` — Foundry's newer, lighter-weight way to define an agent's model + instructions + tools
- How `MCPTool` wires a remote MCP server into an agent, including `require_approval` and `allowed_tools`
- Why `project_connection_id` is used instead of embedding the MCP URL and credentials directly in the tool definition
- `tool_choice="auto"` — letting the model decide when to call a tool, vs. forcing/forbidding tool use
- `create_version` — how Foundry's Agent Service supports versioned agent updates rather than one-shot creation

### Imports and environment-driven configuration

As before, every hardcoded ID from the original script becomes an environment variable, loaded via `python-dotenv`.

In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")

AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "CloudXeus-Invoice-Intelligence-Agent")
DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT")

SEARCH_SERVICE_ENDPOINT = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
KNOWLEDGE_BASE_NAME = os.getenv("AZURE_SEARCH_KNOWLEDGE_BASE_NAME", "kb-cloudxeus-course-products")
PROJECT_CONNECTION_NAME = os.getenv(
    "AZURE_AI_PROJECT_MCP_CONNECTION_NAME", "cloudxeus-product-kb-mcp-connection"
)


### Building the MCP endpoint URL and the project client

This is the exact same `mcp_endpoint` string constructed in `02_project_mcp_conn.ipynb` — it must match, since it's how we (conceptually) reach the same knowledge base, even though at agent-creation time the actual call is routed through the `project_connection_id` below, not this raw URL directly.

In [ ]:
mcp_endpoint = (
    f"{SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    "?api-version=2026-05-01-preview"
)

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)


### Defining the MCP tool

`MCPTool` describes a remote MCP server the agent can call as a tool. `server_label` is a human-readable identifier the model sees; `server_url` is the raw MCP endpoint (kept here mostly for reference/debugging — the live call is authenticated via `project_connection_id`, which tells Foundry to use the managed-identity connection we registered in the previous notebook rather than any credentials embedded here). `allowed_tools` **allow-lists** which specific MCP tools on that server the agent may invoke — here, only `knowledge_base_retrieve` — and `require_approval="never"` means the agent can call it autonomously, with no human-in-the-loop confirmation step.

💡 **Exam tip (AI-102):** MCP tool integrations in Foundry support a `require_approval` setting (commonly `never` / `always`, and sometimes per-tool granularity) that controls whether tool calls need human sign-off before executing — this is the main safety lever for agentic tool use, and it's a natural target for exam questions about "how do you ensure a human reviews an agent's actions before they execute." Similarly, `allowed_tools` is a defense-in-depth allow-list: even if the MCP server exposes many tools, the agent is restricted to only the ones named here.

🔄 **Alternatives:** For a document-processing pipeline like this one, you could swap the MCP-based knowledge base tool for a **File Search** tool backed by an uploaded vector store (as shown in `11_azure_ai_foundry/07_knowledge_rag/`) if you don't need a live, externally-managed search index — simpler to set up, but less suited to a knowledge base that changes independently of the agent.

In [ ]:
knowledge_tool = MCPTool(
    server_label="knowledge-base",
    server_url=mcp_endpoint,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME,
)


### Creating (or versioning) the agent

`client.agents.create_version` registers a new version of the named agent with the given `PromptAgentDefinition` — model, system instructions, tools, and `tool_choice`. Re-running this cell after editing the instructions produces a new version of the *same* agent (identified by `AGENT_NAME`), rather than a brand-new, differently-named agent — that's what lets `04_invoke_agent.ipynb` keep referencing it by a stable name across iterations.

The system instructions are deliberately detailed: they tell the agent what kind of data it will receive (invoice fields extracted upstream), when it must consult the knowledge base tool (product explanations), what to say when it can't find something (say so — don't guess), and how to tailor tone for finance vs. sales vs. customer-facing summaries.

💡 **Exam tip (AI-102):** `tool_choice="auto"` lets the model itself decide, turn by turn, whether a tool call is needed — contrast with forcing a specific tool every turn, or disabling tools entirely. Grounding instructions like "if the knowledge base does not contain enough information, say that you do not know" are a practical mitigation against hallucination/ungrounded answers, a recurring theme in the AI-102 responsible-AI objectives.

🔄 **Alternatives:** Instead of one agent with broad instructions covering finance, sales, and customer-facing summaries, you could define three narrower **connected agents** (see `11_azure_ai_foundry/06_connected_agents/`), each specialized for one audience, and orchestrate between them — trades a larger, more complex single prompt for more, simpler, composable agents.

In [ ]:
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=DEPLOYMENT_NAME,
        instructions=(
            "You are the CloudXeus Invoice Intelligence Agent. "
            "Your role is to help internal CloudXeus teams understand customer invoices "
            "and connect invoice line items to official CloudXeus product information. "

            "You will receive invoice data extracted using Azure Content Understanding. "
            "Use the invoice data to identify the customer, invoice number, invoice date, "
            "due date, line items, quantities, unit prices, taxes, and total amount. "

            "Use the knowledge base tool to answer questions about CloudXeus products. "
            "The knowledge base contains official CloudXeus product-sheet content. "

            "When explaining product features, benefits, or product suitability, "
            "you must use the knowledge base tool. "

            "If the knowledge base does not contain enough information to answer, "
            "say that you do not know based on the available product sheets. "

            "If an invoice line item does not clearly match a product in the knowledge base, "
            "say that the product could not be confidently matched. "

            "For finance summaries, focus on invoice number, customer, dates, totals, taxes, "
            "and any product mismatches. "

            "For sales summaries, focus on purchased products, customer value, product benefits, "
            "and recommended follow-up material. "

            "For customer-facing explanations, use clear and simple business language. "
            "Always be professional, concise, and transparent about missing information."
        ),
        tools=[knowledge_tool],

        # The model decides when to call the knowledge base tool.
        tool_choice="auto",
    ),
)

print("Agent created:")
print(f"  ID      : {agent.id}")
print(f"  Name    : {agent.name}")
print(f"  Version : {agent.version}")


## Summary

We registered a persistent, named Prompt Agent (`CloudXeus-Invoice-Intelligence-Agent`) in the Foundry Agent Service, equipped with an MCP tool that lets it call out to the CloudXeus product knowledge base whenever it needs product details, and detailed instructions tailored to invoice reconciliation across finance, sales, and customer-facing use cases. The printed `agent.id` / `agent.name` / `agent.version` confirm the agent (and this particular version of its definition) now exists in the project. Nothing has been *invoked* yet — that happens in `04_invoke_agent.ipynb`, which looks this agent up **by the same `AGENT_NAME`** via the `agent_reference` pattern.

## Try It Yourself

1. **Easy:** Change `require_approval` to `"always"` and re-run — read the SDK/Foundry docs (or the `04_invoke_agent.ipynb` behavior) to see how invocation now differs when a human approval step is required before a tool call executes.
2. **Intermediate:** Add a second, narrower `allowed_tools` entry if your knowledge base MCP server exposes more than one tool (e.g. a listing tool alongside `knowledge_base_retrieve`), and adjust the instructions to tell the agent when to use each.
3. **Advanced:** Split the single agent's instructions into three separate agents (finance-focused, sales-focused, customer-facing), each created with its own `create_version` call and a distinct `AGENT_NAME`, then sketch (in a markdown cell) how you'd orchestrate between them using the `ConnectedAgentTool` pattern from `11_azure_ai_foundry/06_connected_agents/`.